<img src="https://fedesarrollo.org.co/logo-Fede.png" alt="Fedesarrollo" style="float: right; width: 300px; height: auto;">
<img src="https://upload.wikimedia.org/wikipedia/commons/f/f9/Logo_BID_Espa%C3%B1ol.png" alt="Banco Interamericano de Desarrollo" style="float: right; width: 220px; height: auto;">

# Base de comercio y controles para la estimación de auges exportadores

Editor: Juan Diego Heredia Niño - jheredia@fedesarrollo.org.co

Septiembre 2026

### Importar librerías

La siguiente celda marca el inicio de un nuevo bloque de trabajo independiente:

**Librerías utilizadas:**
- `pandas` y `numpy`: para manipulación y análisis de datos estructurados.
- `glob` y `os`: para listar los archivos anuales de comercio.

In [1]:
import pandas as pd
import numpy as np
import glob
import os

### Configuración de rutas

Carga las rutas del proyecto y las opciones globales desde `config.py`, instalado como
paquete editable en la raíz del proyecto.

In [2]:
from config import raw, temp

### Lectura del deflactor

Carga el deflactor anual preparado en la etapa de descarga, con el que se llevan los valores
corrientes a precios de 2019.

In [3]:
df_deflactor = pd.read_parquet(temp / 'exportaciones' / 'deflactor.parquet')

### Lista de productos en nomenclatura HS 1996

Traduce la lista de productos priorizados, expresada en HS 2012, a la nomenclatura HS 1996 que
usa la base de comercio de largo plazo.

**Pasos realizados:**

1. **Consolidación:** une los productos únicos del ejercicio de caracterización con los de AmCham.

2. **Traducción:** cruza con la tabla de correspondencia HS 2012 a HS 1996.

3. **Colapso por producto:** cuando un mismo código de 1996 recoge varios tipos de oportunidad, los concatena en una sola etiqueta.

In [4]:
(temp / 'exportaciones').mkdir(parents=True, exist_ok=True)

df_productos = pd.read_excel(raw / 'exportaciones' / 'productos' / '2 - Output Final 040725 V02 - EXT.xlsx', sheet_name='Productos únicos')

df_productos['HS6'] = df_productos['HS6'].astype(str).str.zfill(6)

df_productos = df_productos[['HS6','Margen','Industria 3digit']].rename(columns={'HS6': 'hs6_2012','Margen': 'tipo_oportunidad','Industria 3digit': 'industria'})

df_productos_amcham = pd.read_parquet(temp / 'df_amcham.parquet').rename(columns={'HS6': 'hs6_2012', 'Margen': 'tipo_oportunidad', 'Industria': 'industria'})[['hs6_2012','industria','tipo_oportunidad']]

df_products = pd.concat([df_productos, df_productos_amcham], ignore_index=True)[['hs6_2012', 'tipo_oportunidad']].drop_duplicates().reset_index(drop=True)

df_hs96 = pd.read_excel(raw / 'comercio' / 'correlativas' / 'HS 2012 to HS 1996 Correlation and conversion tables.xls', header=6, sheet_name='Conversion HS 2012-HS 1996').drop(columns=['Unnamed: 0', 'Unnamed: 1'])
df_hs96['HS 1996'] = df_hs96['HS 1996'].astype(str).str.zfill(6)
df_hs96['HS 2012'] = df_hs96['HS 2012'].astype(str).str.zfill(6)

df_products = (
    df_products
    .merge(df_hs96, left_on='hs6_2012', right_on='HS 2012', how='left')
    [['HS 1996', 'tipo_oportunidad']]
    .drop_duplicates()
    .rename(columns={'HS 1996': 'hs6_1996'})
    .sort_values(by=['hs6_1996', 'tipo_oportunidad'], ascending=[True, True])
    .groupby('hs6_1996')
    ['tipo_oportunidad']
    .apply(lambda x: ' - '.join(x.unique()))
    .reset_index()
)
# Van sin industria
df_products.to_parquet(temp / 'exportaciones' / 'df_productos.parquet', index=False)

### Base de comercio bilateral de largo plazo

Consolida los archivos anuales de WITS, uno por año.

**Pasos realizados:**

1. **Concatenación:** une los archivos anuales y estandariza el código de producto a seis dígitos.

2. **Filtrado:** conserva las transacciones de al menos cincuenta mil dólares, excluye a Rusia y descarta los socios cuyo comercio total anual no alcanza cien millones.

3. **Renombramiento:** traduce las columnas al esquema de origen y destino que usa la estimación.

4. **Deflactación:** lleva los valores a precios constantes de 2019.

In [5]:
(temp / 'exportaciones').mkdir(parents=True, exist_ok=True)

input_dir = raw / 'exportaciones' / 'comercio'

output_dir = temp / 'exportaciones'
output_file = output_dir / "df_comercio.parquet"

csv_files = glob.glob(os.path.join(input_dir, "*.csv"))

columnas_finales = ['anio','hs6_1996','nombre_o','o_iso3','region_o','ingreso_o',
                    'nombre_d','d_iso3','region_d','ingreso_d','valor_miles_usd']

renombres = {
    "ProductCode":                  "hs6_1996",
    "PartnerISO3":                  "o_iso3",        # país de origen
    "ReporterISO3":                 "d_iso3",        # país de destino
    "TradeValue in 1000 USD":      "valor_miles_usd",
    "Year":                         "anio",
    "Partner Region":               "region_o",
    "Reporter Region":              "region_d",
    "ReporterName":                 "nombre_d",
    "PartnerName":                  "nombre_o",
    'Reporter Income Group':           'ingreso_d',
    'Partner Income Group':            'ingreso_o'
}

# EXCEPCION DE MIGRACION: el original leia los 28 archivos completos, los concatenaba en
# memoria y filtraba sobre el resultado. Son 7,4 GB de texto, que en pandas no caben en los
# 16 GB de la maquina. Aqui se filtra archivo por archivo y se concatena lo que sobrevive.
# El resultado es el mismo: cada archivo contiene exactamente un año, y tanto el groupby de
# TotalTradeValuePerPartner como el drop_duplicates agrupan por Year, de modo que ninguno
# de los dos cruza archivos. Ver tools/markdown/AUDITORIA_MIGRACION.md
dfs = []
for f in csv_files:
    df_anio = pd.read_csv(f, encoding="latin1")
    df_anio['ProductCode'] = df_anio['ProductCode'].astype(str).str.zfill(6)
    df_anio['TotalTradeValuePerPartner'] = df_anio.groupby(['Year', 'PartnerISO3'])['TradeValue in 1000 USD'].transform('sum')
    df_anio = (
        df_anio
        .query("`TradeValue in 1000 USD` >= 50 and ReporterISO3 != 'RUS' and PartnerISO3 != 'RUS' and TotalTradeValuePerPartner >= 100_000")
        .rename(columns=renombres)
        [columnas_finales]
        .drop_duplicates()
    )
    dfs.append(df_anio)
    print(f"{os.path.basename(f)}: {len(df_anio):,} filas")

df = pd.concat(dfs, ignore_index=True).reset_index(drop=True)
del dfs

df = df.merge(df_deflactor, on='anio', how='left')
df['valor_miles_usd'] = df['valor_miles_usd'] * df['deflactor_2019']
df.to_parquet(output_file, index=False)
print(f"\ndf_comercio: {len(df):,} filas")

DataJobID-2915910_2915910_TS04.csv: 475,062 filas


DataJobID-2915902_2915902_TS99.csv: 356,640 filas


DataJobID-2915925_2915925_TS10.csv: 609,197 filas


DataJobID-2915921_2915921_TS08.csv: 618,511 filas


DataJobID-2915934_2915934_TS15.csv: 667,518 filas


DataJobID-2915939_2915939_TS19.csv: 699,798 filas


... (19 outputs left out)

DataJobID-2915932_2915932_TS14.csv: 668,458 filas


DataJobID-2915930_2915930_TS13.csv: 664,343 filas


DataJobID-2915899_2915899_TS95.csv: 291,381 filas



df_comercio: 15,575,422 filas


### Controles gravitacionales

Adjunta a cada par de países y año el índice real de precios de alimentos, la proporción de
tierra cultivable del país de origen, la distancia bilateral y el índice de similitud cultural.

In [6]:
(temp / 'exportaciones').mkdir(parents=True, exist_ok=True)

df_controles = pd.read_excel(raw / 'exportaciones' / 'controles' / 'Tablas.xlsx', sheet_name=None)

(
    df[['o_iso3', 'd_iso3', 'anio']].drop_duplicates()
    .merge(df_controles['FAO'], how='left', left_on=['anio'], right_on=['Año']).drop(columns=['Año','Nominal_Food_Index']).rename(columns={'Real_Food_Index': 'food_index'})
    .merge(df_controles['Tierra'], how='left', left_on=['o_iso3', 'anio'], right_on=['OrigenISO3', 'Año']).drop(columns=['OrigenISO3', 'Año']).rename(columns={'Tierra_Cultivable_%': 'tierra_cultivable_o'})
    .merge(df_controles['Distancia'], how='left', left_on=['o_iso3', 'd_iso3'], right_on=['OrigenISO3', 'DestinoISO3']).drop(columns=['OrigenISO3', 'DestinoISO3']).rename(columns={'Distancia_km': 'distancia'})
    .merge(df_controles['Cultura'], how='left', left_on=['o_iso3', 'd_iso3'], right_on=['OrigenISO3', 'DestinoISO3']).drop(columns=['OrigenISO3', 'DestinoISO3']).rename(columns={'Cultural_Similarity_Index': 'cultura_similaridad'})
).to_parquet(temp / 'exportaciones' / 'df_controles.parquet', index=False)

### Participaciones de mercado de los países de referencia

Construye la participación de México, Costa Rica y República Dominicana en las importaciones
estadounidenses de cada producto, que es el insumo del ejercicio de proporciones de éxito.

In [7]:
(temp / 'exportaciones').mkdir(parents=True, exist_ok=True)

df_mex = pd.read_csv(raw / "exportaciones" / "otros_paises" / "DataJobID-2935804_2935804_MX.csv", encoding="latin1")[['ReporterISO3', 'ProductCode','PartnerISO3', 'Year','TradeValue in 1000 USD']]
df_mex['ProductCode'] = df_mex['ProductCode'].astype(str).str.zfill(6)
df_cri = pd.read_csv(raw / "exportaciones" / "otros_paises" / "DataJobID-2935832_2935832_CR.csv", encoding="latin1")[['ReporterISO3', 'ProductCode','PartnerISO3', 'Year','TradeValue in 1000 USD']]
df_cri['ProductCode'] = df_cri['ProductCode'].astype(str).str.zfill(6)
df_dom = pd.read_csv(raw / "exportaciones" / "otros_paises" / "DataJobID-2938474_2938474_RD.csv", encoding="latin1")[['ReporterISO3', 'ProductCode','PartnerISO3', 'Year','TradeValue in 1000 USD']]
df_dom['ProductCode'] = df_dom['ProductCode'].astype(str).str.zfill(6)
df_oferta = (
    pd.concat([df_mex,df_cri, df_dom], ignore_index=True)
    .sort_values(by=['Year', 'ReporterISO3', 'ProductCode', 'PartnerISO3'], ascending=True)
    .merge(df_deflactor, left_on='Year', right_on='anio', how='left')
    .drop(columns=['anio'])
)
df_oferta['TradeValue in 1000 USD'] = df_oferta['TradeValue in 1000 USD'] * df_oferta['deflactor_2019']
df_oferta.drop(columns=['deflactor_2019'], inplace=True)

df_demanda = (
    pd.read_csv(raw / "exportaciones" / "otros_paises" / "DataJobID-2935826_2935826_USA.csv", encoding="latin1")
    [['ReporterISO3', 'ProductCode', 'Year','TradeValue in 1000 USD']]
    .rename(columns={'TradeValue in 1000 USD': 'Total TradeValue in 1000 USD'})
    .merge(df_deflactor, left_on='Year', right_on='anio', how='left')
    .drop(columns=['anio'])
    )
df_demanda['ProductCode'] = df_demanda['ProductCode'].astype(str).str.zfill(6)
df_demanda['Total TradeValue in 1000 USD'] = df_demanda['Total TradeValue in 1000 USD'] * df_demanda['deflactor_2019']
df_demanda.drop(columns=['deflactor_2019'], inplace=True)

df_paises = df_oferta.merge(df_demanda, left_on=['PartnerISO3', 'ProductCode', 'Year'], right_on=['ReporterISO3', 'ProductCode', 'Year'], how='left', suffixes=('', '_dda')).drop(columns=['ReporterISO3_dda'])
df_paises['share'] = (df_paises['TradeValue in 1000 USD'] / df_paises['Total TradeValue in 1000 USD'])

df_paises.to_parquet(temp / "exportaciones" / "df_paises.parquet", index=False)